# Experiment Runner


## Configuration


In [ ]:
import os
import random
import numpy as np
import torch

config = {"input_len": 400, "model_output_len": 100, "eval_output_len": 200, "metric_output_len": 200, "stride": 100, "state_cols": ["heave", "surge"], "target_cols": ["heave", "surge"], "report_col": "heave", "downsample_factor": 1, "use_pca": False, "pca_n_components": 0.95, "experiment_name": "all_db", "model_names": ["BiLSTM", "TCN", "Transformer"], "hidden_size": 16, "num_layers": 2, "nhead": 2, "dim_feedforward": 64, "dropout": 0.0, "lr": 0.001, "epoch": 10000, "batch_size": 64, "grad_clip": 1.0, "patience": 10, "tcn_channels": [32,32,64], "tcn_kernel_size": 3, "tcn_dropout": 0.1, "loss_type": "db", "peak_percentile": 90, "t_peak": None, "heave_loss_weight": 1.0, "lambda_peak": 0.05, "fu": 1.4, "fo": 1.4, "lambda_tildeq": 0.02, "lambda_amp": 0.05, "lambda_fft_phase": 0.01, "lambda_corr": 0.10, "save_checkpoints": True, "checkpoint_root": "./loss_checkpoints", "use_optuna_best_params": False, "run_optuna": False, "optuna_n_trials": 30, "optuna_epoch": 30, "optuna_patience": 5, "results_dir": "./results_ar_eval_200", "seed": 42}


## Derived Configuration


In [ ]:
config["target_dim"] = len(config["target_cols"])
config["input_dim"] = len(config["state_cols"])
config["report_idx"] = config["target_cols"].index(config["report_col"])
config["heave_idx"] = config["report_idx"]
config["experiment_dir"] = os.path.join(config["results_dir"], config["experiment_name"])
config["optuna_best_params_path"] = os.path.join(config["experiment_dir"], "optuna_best_params_by_model.json")
config["checkpoint_dir"] = os.path.join(config["checkpoint_root"], config["experiment_name"])
os.makedirs(config["experiment_dir"], exist_ok=True)
os.makedirs(config["checkpoint_dir"], exist_ok=True)
if config["input_len"] <= 0 or config["model_output_len"] <= 0 or config["eval_output_len"] <= 0: raise ValueError("Lengths must be positive.")
if config["eval_output_len"] % config["model_output_len"]: raise ValueError("eval_output_len must be a multiple of model_output_len.")
if config["metric_output_len"] > config["eval_output_len"]: raise ValueError("metric_output_len must not exceed eval_output_len.")
if config["report_col"] not in config["target_cols"]: raise ValueError("report_col must be in target_cols.")
if not set(config["target_cols"]).issubset(config["state_cols"]): raise ValueError("target_cols must be in state_cols.")
if not config["model_names"] or config["epoch"] <= 0 or config["batch_size"] <= 0: raise ValueError("Invalid model_names, epoch, or batch_size.")
if config["loss_type"] not in {"mse","ep","dilate","tildeq","db","kmbdf","cp"}: raise ValueError("Unsupported loss_type.")


## Reproducibility


In [ ]:
seed=config["seed"]
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False


## Pipeline Execution


In [ ]:
from IPython.utils.capture import capture_output
from pathlib import Path
if not all(Path(f"{i:02d}_{name}.ipynb").exists() for i,name in [(1,"data_preparation"),(2,"models_and_losses"),(3,"training_and_optuna"),(4,"ar_evaluation_and_figures")]):
    raise FileNotFoundError("Run this notebook with notebooks/ as the working directory.")
print("Experiment started")
for key in ["input_len","model_output_len","eval_output_len","state_cols","target_cols","report_col","loss_type","model_names","run_optuna","use_optuna_best_params","experiment_dir"]: print(f"{key}: {config[key]}")
print("AR ?? ??:",config["eval_output_len"]//config["model_output_len"])
with capture_output():
    %run -i 01_data_preparation.ipynb
    %run -i 02_models_and_losses.ipynb
    %run -i 03_training_and_optuna.ipynb
%run -i 04_ar_evaluation_and_figures.ipynb
print("Experiment completed")


## Final Evaluation Output
